# Lab 3.4.2 — Watch a perceptron learn AND

**HO-3.4.2 (H1): Experience the implementation of a perceptron.**

You will run a small, fully supplied implementation and watch its weights and bias change until all four AND inputs are classified correctly. Allow about **25–35 minutes**.

Read syllabus sections **3.4 and 3.4.1** first. No previous notebook, dataset, or Python programming experience is required. The arithmetic uses multiplication, addition, and comparisons with zero.

Work from top to bottom using **Shift+Enter**. Write in the three **Your observation** Markdown cells: double-click the cell, replace the placeholder, then press Shift+Enter. Two small experiments ask you to change a supplied number. All code works as provided; there are no hidden answers or code-completion puzzles.

This is personal study material, not an official ISTQB exercise. The syllabus supplies the objective and the weights → bias → activation → learning explanation. The precise activation rule, update formula, initial values, and displays below are implementation choices taught here.

## 1. The function we want to learn

Think of a lamp controlled by two switches. A switch is **0** when off and **1** when on. The lamp should turn on **only when both switches are on**. This is the AND function.

| First switch `x1` | Second switch `x2` | Correct lamp output `target` |
| --- | --- | --- |
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

These four labeled examples are the entire problem. We reuse them to observe learning and to check the resulting model. There is no separate test set: we are demonstrating a known four-case function, not estimating performance on an unseen population.

The model will receive the two switch values and learn from these targets. Its prediction code will not contain the AND rule.

In [ ]:
from IPython.display import display
from lab_display import table, show_cases, show_trace, show_history, show_boundary

# Fixed order: one epoch will visit these four examples from top to bottom.
examples = [(0, 0, 0), (0, 1, 0), (1, 0, 0), (1, 1, 1)]
print("Ready. Each example contains (first switch, second switch, target).")

## 2. How one perceptron makes a prediction

Our perceptron has two **weights**, `w1` and `w2`, and one **bias**, `b`. A weight controls the contribution from its input. The bias is added regardless of the inputs.

First calculate a score:

`score = w1 × x1 + w2 × x2 + b`

Then apply this **activation function**:

- If the score is **zero or positive**, predict **1**.
- If the score is **negative**, predict **0**.

For example, with weights `(1, 1)`, bias `-2`, and inputs `(1, 0)`, the score is `1 × 1 + 1 × 0 - 2 = -1`, so the output is 0. A score of exactly zero would give 1. This equality rule matters throughout the lab.

We start with weights `(0, 0)` and bias `0` to make every update easy to follow and repeat. The syllabus describes small random starting weights as typical; fixed zeros are a deliberate choice for this single-perceptron demonstration, not advice for initializing a deep network.

Run the next cell to see what this untrained model does with all four inputs. **No parameters change during this check.**

In [ ]:
def predict(x1, x2, weights, bias):
    score = weights[0] * x1 + weights[1] * x2 + bias
    return int(score >= 0)

initial_weights = (0, 0)
initial_bias = 0
show_cases(examples, initial_weights, initial_bias, predict,
           "Before training: weights (0, 0), bias 0")

## 3. How a mistake changes the model

Training visits one labeled example at a time. For each example, it predicts using the **current** weights and bias, then calculates:

`error = target - prediction`

| Error | What happened | Correction |
| --- | --- | --- |
| +1 | Predicted 0 but needed 1 | Increase the score for this example |
| -1 | Predicted 1 but needed 0 | Decrease the score for this example |
| 0 | Prediction was correct | Leave the parameters unchanged |

We use the following perceptron update rule, with a **learning rate** of 1. The learning rate is a multiplier controlling the size of a correction; we keep it fixed.

`new w1 = old w1 + error × x1`  
`new w2 = old w2 + error × x2`  
`new b  = old b  + error`

All three updates use the same error from the prediction made **before** the update. An input of 0 gives no change to its weight. The bias can change even when both inputs are 0.

For the first example, `(0, 0) → 0`, the initial score is 0, so the model predicts 1. The error is `0 - 1 = -1`. Both weights stay 0 because both inputs are 0; the bias becomes -1.

An **epoch** is one complete pass through all four examples. Each row starts with the parameters left by the preceding row. A correction can help the current example while making another example wrong again, so learning may need several epochs. This is a direct perceptron update; you do not need backpropagation or calculus for this exercise.

In [ ]:
# Supplied implementation: read the comments, then run without edits.
def train_perceptron(examples, max_epochs=20):
    weights = [0, 0]
    bias = 0
    trace = []
    history = []

    for epoch in range(1, max_epochs + 1):
        updates = 0
        for x1, x2, target in examples:
            before_weights = tuple(weights)
            before_bias = bias
            score = weights[0] * x1 + weights[1] * x2 + bias
            prediction = predict(x1, x2, weights, bias)
            error = target - prediction

            # Learning rate = 1. Change parameters after this prediction.
            weights[0] += error * x1
            weights[1] += error * x2
            bias += error
            updates += int(error != 0)
            trace.append({
                "epoch": epoch, "inputs": (x1, x2), "target": target,
                "before_weights": before_weights, "before_bias": before_bias,
                "score": score, "prediction": prediction, "error": error,
                "after_weights": tuple(weights), "after_bias": bias,
            })

        # Freeze the parameters and check ALL four cases without updating.
        wrong = sum(predict(x1, x2, weights, bias) != target
                    for x1, x2, target in examples)
        history.append({
            "epoch": epoch, "weights": tuple(weights), "bias": bias,
            "updates": updates, "wrong": wrong,
        })
        if wrong == 0:
            break

    return trace, history

first_trace, first_history = train_perceptron(examples, max_epochs=1)
show_trace(first_trace, epoch=1)

### Observation 1 — Follow the effect of one pass

The table above shows all four training steps, including parameters before and after each step. The **Weights before** and **Bias before** columns contain the values used to calculate that row's score. A changed bias or weight is used on the **next** row.

Run the following cell to check the four inputs again using the parameters at the **end of epoch 1**. This check makes no further updates.

Then write **2–3 sentences** using both tables: identify an input that was correct when visited during training but is wrong in the end-of-epoch check. Name the later training input whose update caused the change, and use the before/after weights or bias to explain it. This helps explain why one complete pass may not be enough. You may use either qualifying input; there is no required wording.

In [ ]:
first_state = first_history[-1]
show_cases(examples, first_state["weights"], first_state["bias"], predict,
           "After epoch 1: check without further learning")

**Your observation 1**

_Write 2–3 sentences here after comparing the two tables._

## 4. Continue until all four answers are correct

Run the next cell to train **from the same starting values again**, this time allowing up to 20 epochs. It stops as soon as an end-of-epoch check has no wrong predictions.

The summary separates two counts:

- **Updates during pass:** how many examples caused a correction while the parameters were changing.
- **Wrong after pass:** how many of the four examples are wrong when checked with the fixed parameters at the end of that epoch.

These counts answer different questions. A pass can contain corrections and still finish with no wrong predictions. We judge the final state by **Wrong after pass**. Its values need not decrease every epoch.

Here, “zero error” means that the final parameters predict all four targets correctly. We count wrong predictions rather than adding signed errors, which could cancel each other.

In [ ]:
trace, history = train_perceptron(examples, max_epochs=20)
show_history(history)
final_state = history[-1]

if final_state["wrong"] == 0:
    print(f"All four cases correct after epoch {final_state['epoch']}.")
else:
    print("The epoch limit was reached with mistakes remaining.")
print(f"Final weights: {final_state['weights']}; final bias: {final_state['bias']}")

## 5. See what the learned numbers do

With two inputs, we can draw each switch combination as a point. The horizontal axis is `x1`; the vertical axis is `x2`.

A **decision boundary** is where the score equals zero. For this perceptron it is a straight line. On one side the score is negative (prediction 0); on the other it is positive (prediction 1). Points **on** the line also receive prediction 1 under our rule. The plot labels each point with its target and prediction, so you do not need to read it by color.

This is the meaning of **linearly separable** in this two-input example: a straight line can separate the points that need output 0 from those that need output 1. The plot includes space between the four points to show the line; only the four switch combinations are part of our task.

Run the next cell with `selected_epoch = 1`. Then replace 1 with the **final epoch number printed above** and run it again. You may inspect intermediate epochs too. Selecting an epoch only displays saved values; it does not retrain or change the final model.

In [ ]:
# Edit this number only: choose an epoch from 1 to len(history).
selected_epoch = 1

if type(selected_epoch) is not int or not 1 <= selected_epoch <= len(history):
    print(f"Choose a whole-number epoch from 1 to {len(history)}, then rerun this cell.")
else:
    state = history[selected_epoch - 1]
    print(f"Epoch {selected_epoch}: weights {state['weights']}, bias {state['bias']}")
    show_cases(examples, state["weights"], state["bias"], predict,
               f"Fixed check after epoch {selected_epoch}")
    show_boundary(examples, state["weights"], state["bias"], predict)

### Observation 2 — Compare the first and final states

Using the case tables for epoch 1 and the final epoch, write **2–3 sentences**: choose one input that was wrong after epoch 1, report its score and prediction in both states, and explain how the change in score corrected it under our activation rule. The diagram is a visual aid; the score table supplies the exact numbers. The epoch 1 table is also preserved in Section 3.

This connects the changing parameters to a specific improvement in behavior, rather than just noting that the total error fell.

**Your observation 2**

_Input chosen: … Score and prediction after epoch 1: … Score and prediction at the end: … What changed: …_

## 6. A small experiment: change only the bias

The final weights and bias work **together**. To see the bias's role, we will temporarily keep the learned weights and shift just the bias.

Adding 1 to the bias adds 1 to the score of **every** input. Subtracting 1 subtracts 1 from every score. The weights stay fixed, so the line moves without changing its direction.

First run the cell below with the supplied `bias_shift = 1`. Compare the original and changed predictions, and record the +1 part of Observation 3 below. Then try `bias_shift = -1` and rerun the cell to see the other direction. Rerunning replaces the displayed output, so record each result before changing the number. Each run starts from the original learned bias, so the changes do not accumulate. Neither run trains the model.

Write **2–4 sentences in total** in Observation 3. For each shift, identify one input whose prediction changed and say whether it now turns the lamp on when it should be off, or off when it should be on. Use the scores in the comparison table to explain why. This tests the practical effect of the bias without asking you to invent a new training rule.

In [ ]:
# Try 1 first, then -1. No other edit is needed.
bias_shift = 1

if type(bias_shift) is not int or bias_shift not in (-1, 1):
    print("Use 1 or -1 for this comparison, then rerun the cell.")
else:
    learned_weights = final_state["weights"]
    learned_bias = final_state["bias"]
    changed_bias = learned_bias + bias_shift
    rows = []
    for x1, x2, target in examples:
        old_score = learned_weights[0] * x1 + learned_weights[1] * x2 + learned_bias
        new_score = old_score + bias_shift
        old_prediction = predict(x1, x2, learned_weights, learned_bias)
        new_prediction = predict(x1, x2, learned_weights, changed_bias)
        rows.append([(x1, x2), target, old_score, old_prediction,
                     new_score, new_prediction,
                     "changed" if old_prediction != new_prediction else "same"])
    print(f"Weights fixed at {learned_weights}. Bias: {learned_bias} → {changed_bias}.")
    table(["Inputs", "Target", "Original score", "Original prediction",
           "Changed score", "Changed prediction", "Comparison"], rows)
    show_boundary(examples, learned_weights, changed_bias, predict)

**Your observation 3**

_With bias +1: …_

_With bias -1: …_

## 7. Verify the learned AND function

The next cell returns to the **original learned parameters**, unaffected by the display experiments. It checks each of the four inputs against the truth table from Section 1.

The automated check requires exactly what we set out to demonstrate: all four predicted outputs must match AND. It does not require a particular pair of weights, bias, or epoch count; more than one set of parameters can implement the same function.

In [ ]:
show_cases(examples, final_state["weights"], final_state["bias"], predict,
           "Final verification: original learned model")
predictions = [predict(x1, x2, final_state["weights"], final_state["bias"])
               for x1, x2, target in examples]
targets = [target for x1, x2, target in examples]
assert predictions == targets, "The learned model must match all four AND targets."
print("Verified: the original learned model implements AND for all four inputs.")

## Finished

You have seen the full learning loop: make a prediction, compare it with a known target, update weights and bias, repeat across epochs, and check the resulting behavior with fixed parameters.

Your completed work is this notebook with its outputs and three observations. There is no separate learning log and no additional quiz. Save it with **Ctrl+S**. Keep the starter tag unchanged; a later completion commit can record your answers and outputs.

**Source map**

- CT-AI v2.0 §3.4: the single-layer perceptron and linearly separable binary classification.
- §3.4.1: weights, bias, activation functions, learning from error, and epochs.
- §3.4.2 / HO-3.4.2 (H1): demonstrate a perceptron learning a simple function, modifying weights and bias across epochs until error reaches zero.
- §0.5: H1 is a guided exercise.
- The AND data, explicit update formula, zero initialization, learning rate of 1, score-at-zero convention, and stopping check are supplied implementation choices. They are not additional syllabus requirements.

Local primary source (in the study vault): `../../markdown_syllabus/03_machine_learning.md`. The syllabus is outside this Git repository; the explanations needed to run this exercise are included above.